# Figure 5: accuracy and precision

The cell below is tagged `#| label: fig-precision-output`, so `05-issues.md` embeds its output with `:::{figure} #fig-precision-output`. The simulation is in `cardiac_sim.py` in this folder.

In [ ]:
#| label: fig-precision-output
# Figure 5: accuracy vs precision. Each method is simulated many times with random noise.
# The simulation itself is in cardiac_sim.py (shared with Figures 3 and 4).
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import warnings, logging
# Kaleido (the PNG export for the PDF) can print harmless clean-up warnings on Windows.
warnings.filterwarnings("ignore", message=".*temporary directory.*")
for name in ("choreographer", "kaleido", "logistro"):
    logging.getLogger(name).setLevel(logging.CRITICAL)
import os, sys
sys.path.append(os.path.join(os.getcwd(), "notebooks"))  # find cardiac_sim.py also when run from the book folder
from cardiac_sim import SEQUENCES, simulate, fit

# Interactive figure for the website + PNG snapshot for the PDF (same probe as the template).
try:
    pio.to_image(go.Figure(), format="png")
    pio.renderers.default = "plotly_mimetype+png"
except Exception as exc:
    pio.renderers.default = "plotly_mimetype"
    print(f"Static PNG export unavailable ({type(exc).__name__}); PDF will show the caption only.")

T1 = 1000.0                # native myocardium [ms]
RR = 1000.0                # 60 beats per minute
SNR_VALUES = [20, 40, 80]  # SNR = M0 / standard deviation of the noise
N_REPEATS = 500            # number of simulated scans per method and SNR
COLORS = {"MOLLI": "royalblue", "ShMOLLI": "seagreen", "SASHA": "darkorange"}
rng = np.random.default_rng(seed=1)   # fixed seed, so the figure is the same every time

fig = go.Figure()
for j, snr in enumerate(SNR_VALUES):
    for seq in SEQUENCES:
        _, _, _, samples = simulate(seq, T1, RR)
        estimates = np.array([fit(seq, samples, RR, rng.normal(0, 1 / snr, len(samples)))[2]
                              for _ in range(N_REPEATS)])
        fig.add_trace(go.Histogram(
            x=estimates, xbins=dict(start=500, end=1500, size=10), opacity=0.6,
            marker_color=COLORS[seq], visible=(snr == 40),
            name=f"{seq}: mean {estimates.mean():.0f} ms, SD {estimates.std():.0f} ms"))
fig.add_vline(x=T1, line=dict(color="black", dash="dash"),
              annotation_text="true T1", annotation_position="top")

steps = []
for j, snr in enumerate(SNR_VALUES):
    visible = [False] * (len(SNR_VALUES) * len(SEQUENCES))
    visible[j * len(SEQUENCES):(j + 1) * len(SEQUENCES)] = [True] * len(SEQUENCES)
    steps.append(dict(method="update", args=[{"visible": visible}], label=str(snr)))

fig.update_layout(
    barmode="overlay", height=600, template="plotly_white",
    sliders=[dict(active=SNR_VALUES.index(40), steps=steps,
                  currentvalue=dict(prefix="SNR: "), pad=dict(t=50))],
    xaxis_title="Fitted T1 [ms]", yaxis_title="Number of simulated scans",
    xaxis_range=[600, 1400], margin=dict(t=40, b=200),
    # legend at the very bottom of the figure, below the slider, with space in between
    legend=dict(orientation="h", yref="container", yanchor="bottom", y=0.01, x=0),
)
fig
